# Data Cleaning & Transformation

This notebook transforms the raw Bangalore Rapido Ride dataset into a clean, structured,
and analysis-ready format.

Key steps:
- Handling missing values using business logic
- Converting date and time columns
- Removing duplicates
- Standardizing categorical values
- Feature engineering

Special attention is given to cancelled rides, where missing values represent valid business scenarios.

## Loading Raw Dataset

The raw dataset is loaded from the extraction phase.

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/raw/rides_data.csv")
df.head()

,services,date,time,ride_status,source,destination,duration,ride_id,distance,ride_charge,misc_charge,total_fare,payment_method
0,cab economy,2024-07-15,08:30:40.542646,completed,Balagere Harbor,Harohalli Nagar,39,RD3161218751875354,27.21,764.83,31.51,796.34,Amazon Pay
1,auto,2024-07-05,23:36:51.542646,completed,Basavanagudi 3rd Block,Bikasipura 1st Stage,89,RD8171514284594096,34.03,314.83,49.52,364.35,Paytm
2,auto,2024-07-23,11:05:37.542646,cancelled,Babusapalya Cove,Kothaguda Terrace,25,RD9376481122237926,20.24,NaN,NaN,NaN,NaN
3,cab economy,2024-06-24,08:45:10.542646,completed,Mahadevapura Mews,Kanakapura Arc,89,RD3676889143182765,31.17,484.73,15.84,500.57,QR scan
4,cab economy,2024-07-15,00:26:44.542646,completed,Ganganagar Cove,Basaveshwaranagar Colony,95,RD6639410275948084,27.21,663.50,14.13,677.63,Amazon Pay


## Data Inspection

We examine the dataset to identify:
- Missing values
- Incorrect data types
- Data inconsistencies

This step defines the cleaning strategy.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   services        50000 non-null  str    
 1   date            50000 non-null  str    
 2   time            50000 non-null  str    
 3   ride_status     50000 non-null  str    
 4   source          50000 non-null  str    
 5   destination     50000 non-null  str    
 6   duration        50000 non-null  int64  
 7   ride_id         50000 non-null  str    
 8   distance        50000 non-null  float64
 9   ride_charge     44964 non-null  float64
 10  misc_charge     44964 non-null  float64
 11  total_fare      44964 non-null  float64
 12  payment_method  44964 non-null  str    
dtypes: float64(4), int64(1), str(8)
memory usage: 5.0 MB


In [3]:
df.isnull().sum()

services             0
date                 0
time                 0
ride_status          0
source               0
destination          0
duration             0
ride_id              0
distance             0
ride_charge       5036
misc_charge       5036
total_fare        5036
payment_method    5036
dtype: int64

## Handling Missing Values

Missing values are observed in:
- ride_charge
- misc_charge
- total_fare
- payment_method

These missing values correspond to cancelled rides.

Handling strategy:
- Cancelled rides → fare values set to 0, payment set to "Not Applicable"
- Completed rides → total_fare recalculated as ride_charge + misc_charge

This ensures logical consistency and avoids data loss.

In [4]:
cancelled_mask = df['ride_status'] == 'cancelled'
# Set values for cancelled rides
df.loc[cancelled_mask, ['ride_charge', 'misc_charge', 'total_fare']] = 0
df.loc[cancelled_mask, 'payment_method'] = "Not Applicable"

In [5]:
# Recalculate total fare for completed rides
completed_mask = df['ride_status'] == 'completed'
df.loc[completed_mask, 'total_fare'] = (
    df.loc[completed_mask, 'ride_charge'] +
    df.loc[completed_mask, 'misc_charge']
)

## Date & Time Conversion

- The `date` column is converted into a standard datetime format.
- The `time` column has microsecond precision (e.g., `08:30:40.542646`). We explicitly define the format string to prevent pandas from guessing, which avoids parsing warnings and speeds up execution.

In [6]:
# Convert Date
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date']) # Drop any rows where date parsing failed

# Convert Timestamp efficiently using exact format
df['time_stamp'] = pd.to_datetime(
    df['date'].astype(str) + ' ' + df['time'].astype(str),
    errors='coerce'
)

df = df.dropna(subset=['time_stamp'])

## Removing Duplicates

Duplicate rows based on `ride_id` are dropped.

In [7]:
df = df.drop_duplicates(subset=['ride_id'], keep='first')

## Data Standardization

- Categorical values are standardized (lowercased, stripped).
- Service types with spaces are replaced with underscores.

In [8]:
# Standardise all categorical columns
cols_to_fix = ['services', 'ride_status', 'payment_method', 'source', 'destination']

for col in cols_to_fix:
    df[col] = df[col].astype(str).str.lower().str.strip()

# Map the service names for consistency
df['services'] = df['services'].replace({
    'bike lite': 'bike_lite',
    'cab economy': 'cab_economy'
})

## Feature Engineering

New features are created to enhance analysis:

New features are created to enhance temporal and business analysis:
- `year`, `month`, `day_of_week`
- `hour`,
- `peak_hour` → Identifies high demand periods (8-10 AM, 5-7 PM)
- `is_weekend` → Flag for Saturday/Sunday

In [9]:
# Date features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.day_name()

# Time features
df['hour'] = df['time_stamp'].dt.hour

# Business features
df['peak_hour'] =df['peak_hour'] = df['hour'].isin([8, 9, 10, 17, 18, 19]).astype(int)
df['completed'] = (df['ride_status'] == 'completed').astype(int)
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

## Anomaly Detection & Advanced Metrics

We calculate speed and efficiency.
*Note: Because this dataset appears to have uniformly random distributions for distance (1-50km) and duration (10-120min), applying strict real-world physics (like capping speed at 80km/h) will arbitrarily delete valid synthetic data. We will flag anomalies rather than drop them all, but remove mathematically impossible ones (distance <= 0).*

In [10]:
# Calculate advanced metrics (only for completed rides to avoid division by zero on bad data)
df['avg_speed_kmh'] = df['distance'] / (df['duration'] / 60)
df['fare_per_km'] = np.where(df['distance'] > 0, df['total_fare'] / df['distance'], 0)
df['ride_efficiency'] = np.where(df['duration'] > 0, df['distance'] / df['duration'], 0)

# Filter out strictly impossible mathematical scenarios
# (e.g., negative/zero distance or duration on completed rides)
invalid_rides = df[
    (df['ride_status'] == 'completed') &
    ((df['distance'] <= 0) | (df['distance'] > 100) | (df['duration'] <= 0))
]

df = df.drop(invalid_rides.index)
print(f"Dropped {len(invalid_rides)} logically impossible records.")

Dropped 0 logically impossible records.


## Post-Cleaning Validation

After handling missing values, we verify that:

- No critical null values remain
- All numerical fields are complete
- Dataset is consistent for analysis

This ensures reliability of downstream analysis and dashboard metrics.

In [11]:
df.isnull().sum()

services           0
date               0
time               0
ride_status        0
source             0
destination        0
duration           0
ride_id            0
distance           0
ride_charge        0
misc_charge        0
total_fare         0
payment_method     0
time_stamp         0
year               0
month              0
day_of_week        0
hour               0
peak_hour          0
completed          0
is_weekend         0
avg_speed_kmh      0
fare_per_km        0
ride_efficiency    0
dtype: int64

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 24 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   services         50000 non-null  str           
 1   date             50000 non-null  datetime64[us]
 2   time             50000 non-null  str           
 3   ride_status      50000 non-null  str           
 4   source           50000 non-null  str           
 5   destination      50000 non-null  str           
 6   duration         50000 non-null  int64         
 7   ride_id          50000 non-null  str           
 8   distance         50000 non-null  float64       
 9   ride_charge      50000 non-null  float64       
 10  misc_charge      50000 non-null  float64       
 11  total_fare       50000 non-null  float64       
 12  payment_method   50000 non-null  str           
 13  time_stamp       50000 non-null  datetime64[us]
 14  year             50000 non-null  int32         
 

In [13]:
df.to_csv("../data/processed/cleaned_rides.csv", index=False)

## Data Quality Summary

- 10% missing values due to cancelled rides (handled using business logic)
- Fare inconsistencies corrected using recalculation
- Invalid and unrealistic records removed
- Categorical data standardized
- New features engineered for analysis

Final dataset is clean, consistent, and analysis-ready.